# Introduction to the Weights Quantization

This notebook made by [n.luneva](https://github.com/lwtztea) 🤗

## Base Theory

https://arxiv.org/pdf/2106.08295

### What is Quantization?

Weight quantization is a technique used to reduce the precision of the model weights. This process involves converting the weights from high-precision formats (e.g., 32-bit floating-point numbers) to lower-precision formats (e.g., 8-bit integers or even binary values).

### How It Works?

To move from floating-point to the efficient fixed-point operations, we need a scheme for converting floating-point vectors to integers. A floating-point vector $\mathbf{x}$ can be expressed approximately as a scalar multiplied by a vector of integer values:

$$\hat{\mathbf{x}} = s_{\mathbf{x}} \cdot \mathbf{x}_{int} \approx \mathbf{x}$$

where $s_\mathbf{x}$ is a floating-point scale factor and $x_{int}$ is an integer vector, e.g., INT8. By quantizing the **weights** and **activations*** we can write the
quantized version of the accumulation equation:

$$\hat{\mathbf{A}}_n = \hat{\mathbf{b}}_n + \sum_m \hat{\mathbf{W}}_{n,m}\hat{\mathbf{x}}_m = \hat{\mathbf{b}}_n + \sum_m (s_{\mathbf{w}} \mathbf{W}^{int}_{n,m})(s_{\mathbf{x}} \mathbf{x}^{int}_m) = \hat{\mathbf{b}}_n + s_\mathbf{w} s_{\mathbf{x}} \sum_m \mathbf{W}^{int}_{n,m} \mathbf{x}^{int}_m$$


Note that we used a separate scale factors $s_\mathbf{w}$ for weights and $s_\mathbf{x}$ for activations. This provides flexibility and reduces the quantization error.

***activations** are the outputs of neurons in a neural network after applying an activation function.

<img src = https://raw.githubusercontent.com/lwtztea/ml_pic/357ccf2/week_11/quantization.png width = 1000>

### Why Use Quantization?

- **Reduced Memory Usage.** Lower-precision weights require less memory storage. For example, switching from `float32` to `int8` reduces memory usage by a factor of 4.

- **Faster Inference.** Operations on lower-precision numbers are computationally cheaper. For instance, integer arithmetic is faster than floating-point arithmetic on most hardware.

- **Improved Deployment.** Quantized models are easier to deploy on edge devices with limited computational resources.

### Quantization Types

- **Post-Training Quantization (PTQ).** Applied after the model has been trained. This is simpler but may result in some loss of accuracy.
- **Quantization-aware Training (QAT).** Incorporates quantization effects during training, allowing the model to adapt to the reduced precision. This generally results in better accuracy compared to post-training quantization.


### Linear Quantizations

#### Simmetric Signed Quantization

Maps $0 \rightarrow 0$ and $|\max(w_i)| \rightarrow 2^{b-1} - 1$

$$\mathbf{x}_{int} = \text{clamp}\left(\lfloor \frac{\mathbf{x}}{s} \rceil; -2^{b-1}, 2^{b-1}-1 \right)$$

$$\hat{\mathbf{x}} = s \cdot \mathbf{x}_{int}$$

where:
- $\mathbf{x}$ — original real value
- $\mathbf{x}_{int}$ - quantized value
- $\hat{\mathbf{x}}$ — dequantized value
- $s$ — scale factor
- $b$ — bit-width (number of bits)
- $\lfloor \cdot \rceil$ — rounding operator to the nearest integer
- $\text{clamp}(x; a, c)$ — clamping function

<img src = https://raw.githubusercontent.com/lwtztea/ml_pic/357ccf2/week_11/symmetric_signed.png width = 800>

#### Simmetric Unsigned Quantization

Maps $0 \rightarrow 0$ and $|\max(w_i)| \rightarrow 2^{b} - 1$

$$\mathbf{x}_{int} = \text{clamp}\left(\lfloor \frac{\mathbf{x}}{s} \rceil; 0, 2^b-1 \right)$$

$$\hat{\mathbf{x}} = s \cdot \mathbf{x}_{int}$$

<img src = https://raw.githubusercontent.com/lwtztea/ml_pic/357ccf2/week_11/symmetric_unsigned.png width = 800>

#### Asymmetric Quantization

$$\mathbf{x}_{int} = \text{clamp}\left(\lfloor \frac{\mathbf{x}}{s} + z \rceil; 0, 2^b-1 \right)$$

$$\hat{\mathbf{x}} = s \cdot (\mathbf{x}_{int} - z)$$

where $z$ is the zero-point, which is used to ensure that the real zero after dequantization remains equal to zero. This is important to ensure that common operations like zero padding or ReLU do not induce quantization error.

<img src = https://raw.githubusercontent.com/lwtztea/ml_pic/357ccf2/week_11/asymmetric.png width = 800>

## Base Practice

In [ ]:
from abc import ABC, abstractmethod

import matplotlib.pyplot as plt
import numpy as np

In [ ]:
class BaseSymmetricQuantizer(ABC):
    def __init__(self, num_bits):
        self.num_bits = num_bits
        self.scale = None

        self.q_min = -(2 ** (num_bits - 1))
        self.q_max = 2 ** (num_bits - 1) - 1

    @abstractmethod
    def quantize(self, weights):
        pass

    def dequantize(self, quantized_weights):
        if self.scale is None:
            raise ValueError("Scale is not initialized. Call quantize() first.")

        return quantized_weights * self.scale

In [ ]:
class SymmetricQuantizer(BaseSymmetricQuantizer):
    def quantize(self, weights):
        max_val = np.max(np.abs(weights))
        self.scale = max_val / self.q_max
        normalized_weights = weights / self.scale
        quantized_weights = np.round(normalized_weights)
        quantized_weights = np.clip(quantized_weights, self.q_min, self.q_max)

        return quantized_weights

In [ ]:
M, N = 4, 3  # to display only part of the weights matrix
D_IN, D_OUT = 10, 8  # input and output dimentions
BATCH_SIZE = 1024

In [ ]:
np.random.seed(21)
weights = np.random.randn(D_IN, D_OUT).astype(np.float32) * 0.1
print(f"Original weights:\n {weights[:M, :N]}")

In [ ]:
np.random.seed(21)
activations = np.random.randn(BATCH_SIZE, D_IN).astype(np.float32) * 0.1
print(f"Activations:\n {activations[:2, :M]}")

In [ ]:
quantizer = SymmetricQuantizer(num_bits=8)
quantized_weights = quantizer.quantize(weights)
print(f"Quantized weights:\n {quantized_weights[:M, :N]}")

In [ ]:
true_result = activations @ weights
quantized_result = activations @ (quantized_weights * quantizer.scale)

rmse = np.sqrt(np.square((true_result - quantized_result).mean()))
print(f"Quantization error: {rmse}")

## Activation-aware Weight Quantization (AWQ)

https://arxiv.org/pdf/2306.00978

### Key Idea

AWQ leverages the observation that not all weights are equally important. By analyzing the **activation distribution**, AWQ identifies **salient weights** (those processing more important features) and applies scaling to protect them during quantization.

<img src = https://raw.githubusercontent.com/lwtztea/ml_pic/357ccf2/week_11/awq.png width = 1000>

### Analyzing the quantization error

Consider a group/block of weight $\mathbf{w}$; the linear operation can be written as $y = \mathbf{wx}$, and the quantized counterpart is $y = Q(\mathbf{w})\mathbf{x}$. The quantization function is defined as:

$$Q(\mathbf{w}) = \Delta \cdot \text{Round}(\frac{\mathbf{w}}{\Delta}), \ \Delta = \frac{max(|\mathbf{w}|)}{2^{N-1}}$$

where $N$ is the number of quantization bits, and $\Delta$ is the quantization scale factor.

Multiplying both sides by $\frac{x}{s}$ we obtain:

$$Q(\mathbf{w})\cdot \frac{x}{s} = \Delta \cdot \text{Round}(\frac{\mathbf{w}}{\Delta}) \cdot x \cdot \frac{1}{s}$$

Now consider a weight element $w \in \mathbf{w}$. If we multiply $w$ with $s > 1$ we'll have:

$$Q(ws)\cdot \frac{x}{s} = \Delta' \cdot \text{Round}(\frac{ws}{\Delta'}) \cdot x \cdot \frac{1}{s}$$

where $\Delta'$ is the new quantization scaler after applying $s$.

Authors empirically found that:

1.  The expected error from $\text{Round(·)}$ (denoted as $\text{RoundErr(·)}$) does not change: since the round function maps a floating-point number to an integer, the error is roughly uniformly distributed from $[0,0.5]$, resulting in an average error of $0.25$; i.e., $\text{RoundErr(·)} \sim 0.25$.

2. Scaling up a single element $w$ usually does not change the maximum value from the group $\mathbf{w}$. Therefore we have $\Delta \approx \Delta'$

3. As $\Delta$ and $x$ are represented in FP16, they have no quantization error. Consequently, the quantization error from equations above can be expressed as:

$$\text{Err}(Q(w)x) = \Delta \cdot \text{RoundErr}(\frac{w}{\Delta}) \cdot x$$

$$\text{Err}(Q(w \cdot s)(\frac{x}{s})) = \Delta' \cdot \text{RoundErr}(\frac{ws}{\Delta'}) \cdot x \cdot \frac{1}{s}$$

The ratio of the new error to the original error is $\frac{\Delta'}{\Delta} \cdot \frac{1}{s}$. Given $\Delta' \approx \Delta$ and $s > 1$, **the relative error is smaller for the salient weight $w$**.

<img src = https://raw.githubusercontent.com/lwtztea/ml_pic/8e9f93c/week_11/awq_benchmarking.png width = 1000>

### Searching to scale

Keeping 0.1% of weights selected based on **activation magnitude** in FP16 can improve the quantized performance without a noticeable increase in model size (measured in total bits). But such a mixedprecision data type will make the system implementation difficult. We need to come up with a method to protect the important weights without actually keeping them as FP16.

Formally, we want to optimize the following objective:

$$\mathbf{s^*} = \underset{\mathbf{s}}{\arg \min} \ L(\mathbf{s})$$

$$L(\mathbf{s}) = || Q(\mathbf{W} \cdot \text{diag}(\mathbf{s}))(\text{diag}(\mathbf{s}^{-1} \cdot \mathbf{X}) - \mathbf{W}\mathbf{X} ||$$

where $Q$ means the weight quantization function, $\mathbf{W}$ is the original weights in FP16, $\mathbf{X}$ is the input features cached from a small calibration set, and $\mathbf{s}$ is a per-(input) channel scaling factor.

Authors decided to simplify the task (since the quantization function is not differentiable) by reducing the searching space:

$$\mathbf{s} = \mathbf{s_X}^\alpha, \ \alpha^* = \underset{\mathbf{\alpha}}{\arg \min} \ L(\mathbf{s_X}^\alpha)$$

where $\mathbf{s_X}$ is the average magnitude of activation (per-channel), and they use a single hyperparameter $\alpha \in [0, 1]$ searched via a fast grid search to balance between the protection of salient and non-salient channels (0 means no scaling; 1 corresponds to the most aggressive scaling in the search space).

### Practice

Let's implement a toy example of AWQ

In [ ]:
class AWQSymmetricQuantizer(BaseSymmetricQuantizer):
    def __init__(self, num_bits):
        super().__init__(num_bits)
        self.avg_magnitude = None

    def compute_average_magnitude(self, activations):
        self.avg_magnitude = np.mean(np.abs(activations), axis=0)

    def quantize(self, weights):
        if self.avg_magnitude is None:
            raise ValueError(
                "Average magnitude of activation is not initialized. Call compute_average_magnitude() first."
            )

        scaled_weights = weights * self.avg_magnitude[:, np.newaxis]

        max_val = np.max(np.abs(scaled_weights))
        self.scale = max_val / self.q_max
        normalized_weights = scaled_weights / self.scale
        quantized_weights = np.round(normalized_weights)
        quantized_weights = np.clip(quantized_weights, self.q_min, self.q_max)

        return quantized_weights

In [ ]:
awq_quantizer = AWQSymmetricQuantizer(num_bits=8)
awq_quantizer.compute_average_magnitude(activations)

awq_quantized_weights = awq_quantizer.quantize(weights)
print(f"AWQ quantized weights:\n {awq_quantized_weights[:M, :N]}")

In [ ]:
awq_quantized_result = (activations / awq_quantizer.avg_magnitude) @ (awq_quantized_weights * awq_quantizer.scale)

rmse = np.sqrt(np.square((true_result - awq_quantized_result).mean()))
print(f"Quantization error: {rmse}")

### Evaluating AWQ

Let's look at the dependence of computational error on the number of observations.

In [ ]:
def compute_quantization_error(activations, weights):
    quantizer = SymmetricQuantizer(num_bits=8)
    quantized_weights = quantizer.quantize(weights)

    quantized_result = activations @ (quantized_weights * quantizer.scale)
    return np.sqrt(np.square((activations @ weights - quantized_result).mean()))

In [ ]:
def compute_aqw_quantization_error(activations, weights):
    awq_quantizer = AWQSymmetricQuantizer(num_bits=8)
    awq_quantizer.compute_average_magnitude(activations)
    awq_quantized_weights = awq_quantizer.quantize(weights)

    awq_quantized_result = (activations / awq_quantizer.avg_magnitude) @ (awq_quantized_weights * awq_quantizer.scale)
    return np.sqrt(np.square((activations @ weights - awq_quantized_result).mean()))

In [ ]:
def plot_observations_error(activations, weights, min_observations=20):
    num_observations = list(range(min_observations, BATCH_SIZE))

    wrapped_error_function = lambda n: compute_quantization_error(n, weights)
    errors = [wrapped_error_function(activations[:n]) for n in num_observations]

    wrapped_awq_error_function = lambda n: compute_aqw_quantization_error(n, weights)
    awq_errors = [wrapped_awq_error_function(activations[:n]) for n in num_observations]

    plt.figure(figsize=(10, 6))
    plt.plot(num_observations, errors, c="g", lw=0.6, label="base quantization error")
    plt.plot(num_observations, awq_errors, c="r", lw=0.6, label="awq quantization error")

    plt.xlabel("Number of observations", fontsize=14)
    plt.ylabel("Computational error", fontsize=14)
    plt.legend(fontsize=12)

    plt.grid(True)
    plt.show()

In [ ]:
plot_observations_error(activations, weights)

))))))))))))) I suspect it's due to the random weights initialization.

#### Real Usage Example

*Unfortunately, this part doesn't work.*

We will compare the performance of a pre-trained LLM before and after applying AWQ.

In [ ]:
!pip install autoawq

In [ ]:
import torch
from awq import AutoAWQForCausalLM
from transformers import AutoModelForCausalLM, AutoTokenizer

In [ ]:
MODEL_PATH = "facebook/opt-125m"
model = AutoModelForCausalLM.from_pretrained(MODEL_PATH)
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

In [ ]:
INPUT_TEXT = "The quick brown fox jumps over the lazy dog."
inputs = tokenizer(INPUT_TEXT, return_tensors="pt")

with torch.no_grad():
    outputs = model.generate(inputs.input_ids, max_length=50)

print("Generated Text:", tokenizer.decode(outputs[0], skip_special_tokens=True))

In [ ]:
QUANT_CONFIG = {"zero_point": True, "q_group_size": 128, "w_bit": 4, "version": "GEMM"}
quantized_model = AutoAWQForCausalLM.from_pretrained(MODEL_PATH).cuda()
quantized_model.quantize(tokenizer, quant_config=QUANT_CONFIG, calib_data="OpenAssistant/oasst2", split="train")

### Conclusion

AWQ is a powerful technique for compressing and accelerating large language models. By focusing on salient weights and leveraging activation distributions, AWQ achieves significant reductions in model size and inference time without compromising performance. This makes it ideal for deploying LLMs on edge devices.